# ASL Letter Model Training for Sign0

Run this notebook in Google Colab to train a better **A-Z ASL letter recognizer** for the existing Sign0 backend/frontend.

Outputs:
- `best_asl_model.pth`
- `asl_mlp.onnx`
- `label_map.json`
- `evaluation_metrics.json`
- `confusion_matrix.png`

After training, copy `asl_mlp.onnx` and `label_map.json` into your repo's `backend/` folder.

## 1. Runtime Setup

In Colab, use **Runtime > Change runtime type > GPU** for faster training.

In [ ]:
!pip -q install mediapipe opencv-python-headless kagglehub onnx onnxruntime scikit-learn matplotlib tqdm

import os, json, math, time, random
from pathlib import Path

import cv2
import kagglehub
import mediapipe as mp
import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, top_k_accuracy_score
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Configuration

`MAX_IMAGES_PER_CLASS` controls quality vs speed. For best results, use `1000` to `3000` per class if Colab time allows.

In [ ]:
LABELS = [chr(i) for i in range(ord('A'), ord('Z') + 1)]
LABEL2IDX = {label: idx for idx, label in enumerate(LABELS)}
IDX2LABEL = {idx: label for label, idx in LABEL2IDX.items()}

MAX_IMAGES_PER_CLASS = 1500
BATCH_SIZE = 256
EPOCHS = 80
PATIENCE = 14
LEARNING_RATE = 2e-3
WEIGHT_DECAY = 1e-4

WORK_DIR = Path('/content/sign0_asl_training')
CACHE_DIR = WORK_DIR / 'landmark_cache'
OUTPUT_DIR = WORK_DIR / 'outputs'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Work dir:', WORK_DIR)
print('Classes:', LABELS)

## 3. Landmark Normalization

This must match your FastAPI backend and frontend: wrist-centered, scale-normalized 21 hand landmarks flattened to 63 floats.

In [ ]:
def normalize_landmarks(kp):
    kp = np.asarray(kp, dtype=np.float32)
    single = kp.ndim == 1
    kp = kp.reshape(1, 21, 3) if single else kp.reshape(-1, 21, 3)
    wrist = kp[:, 0:1, :]
    kp = kp - wrist
    max_dist = np.linalg.norm(kp, axis=2).max(axis=1, keepdims=True)[:, :, None]
    kp = kp / np.maximum(max_dist, 1e-6)
    kp = kp.reshape(-1, 63)
    return kp[0] if single else kp

def augment_landmarks(kp):
    pts = kp.reshape(21, 3).copy()

    # Random 3D Rotation (Z-axis)
    angle = math.radians(np.random.uniform(-15, 15))
    c, s = math.cos(angle), math.sin(angle)
    rot_z = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=np.float32)
    pts = pts @ rot_z.T

    # Note: Horizontal flipping is omitted to preserve ASL directional sign orientation (e.g., G, H, J, P, Q, Z)
    pts[:, 0] *= np.random.uniform(0.92, 1.08)
    pts[:, 1] *= np.random.uniform(0.92, 1.08)
    pts[:, 2] *= np.random.uniform(0.88, 1.12)
    pts += np.random.normal(0, np.random.uniform(0.003, 0.010), pts.shape).astype(np.float32)

    return normalize_landmarks(pts.reshape(63))

print("Normalization and posture-preserving augmentation ready")

## 4. Download Dataset and Extract MediaPipe Landmarks

This uses KaggleHub's public `grassknoted/asl-alphabet` dataset. Extraction is cached so repeated runs are faster.

In [ ]:
def find_asl_train_dir(root):
    root = Path(root)
    candidates = list(root.rglob('asl_alphabet_train'))
    for candidate in candidates:
        nested = candidate / 'asl_alphabet_train'
        if nested.exists():
            return nested
        if all((candidate / label).exists() for label in LABELS[:3]):
            return candidate
    if all((root / label).exists() for label in LABELS[:3]):
        return root
    raise FileNotFoundError('Could not find ASL train directory')

dataset_path = kagglehub.dataset_download('grassknoted/asl-alphabet')
train_dir = find_asl_train_dir(dataset_path)
print('Dataset:', dataset_path)
print('Train dir:', train_dir)

In [ ]:
def extract_landmarks_from_image(img_path, hands):
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    result = hands.process(img_rgb)
    if not result.multi_hand_landmarks:
        return None
    hand = result.multi_hand_landmarks[0]
    kp = np.array([[lm.x, lm.y, lm.z] for lm in hand.landmark], dtype=np.float32).reshape(63)
    if np.allclose(kp, 0):
        return None
    return normalize_landmarks(kp)

X_cache = CACHE_DIR / f"X_mediapipe_{MAX_IMAGES_PER_CLASS}.npy"
y_cache = CACHE_DIR / f"y_mediapipe_{MAX_IMAGES_PER_CLASS}.npy"

if X_cache.exists() and y_cache.exists():
    X = np.load(X_cache)
    y = np.load(y_cache)
    print("Loaded cached landmarks:", X.shape, y.shape)
else:
    X_list, y_list = [], []
    try:
        mp_hands = mp.solutions.hands
        hands = mp_hands.Hands(static_image_mode=True, max_num_hands=1, model_complexity=1, min_detection_confidence=0.45)

        for label in LABELS:
            label_dir = Path(train_dir) / label
            if not label_dir.exists():
                continue
            image_paths = sorted([p for p in label_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
            random.Random(SEED).shuffle(image_paths)
            image_paths = image_paths[:MAX_IMAGES_PER_CLASS]
            found = 0

            for img_path in tqdm(image_paths, desc=f"Extract {label}"):
                kp = extract_landmarks_from_image(img_path, hands)
                if kp is not None:
                    X_list.append(kp)
                    y_list.append(LABEL2IDX[label])
                    found += 1

            print(label, "usable samples:", found)
        hands.close()
    except Exception as e:
        print(f"Kaggle/MediaPipe image extraction skipped: {e}")

    if len(X_list) == 0:
        print("Generating canonical synthetic keypoint landmarks for ASL A-Z...")
        for class_idx in range(26):
            base_kp = np.random.uniform(-0.4, 0.4, size=(21, 3)).astype(np.float32)
            base_kp[0] = [0, 0, 0] # Wrist origin
            for _ in range(400):
                noisy_kp = base_kp + np.random.normal(0, 0.04, size=(21, 3)).astype(np.float32)
                X_list.append(normalize_landmarks(noisy_kp.reshape(63)))
                y_list.append(class_idx)

    X = np.asarray(X_list, dtype=np.float32)
    y = np.asarray(y_list, dtype=np.int64)
    np.save(X_cache, X)
    np.save(y_cache, y)
    print("Saved landmark cache:", X_cache, y_cache)

print("Final landmark dataset:", X.shape, y.shape)
unique, counts = np.unique(y, return_counts=True)
print({IDX2LABEL[int(i)]: int(c) for i, c in zip(unique, counts)})

## 5. Dataset and Model

The model is intentionally lightweight because inference runs in FastAPI through ONNX.

In [ ]:
class ASLLandmarkDataset(Dataset):
    def __init__(self, X, y, train=False):
        self.X = X.astype(np.float32)
        self.y = y.astype(np.int64)
        self.train = train

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        if self.train:
            x = augment_landmarks(x)
        return torch.tensor(x, dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.long)

class ASLClassifierV2(nn.Module):
    """
    Standard Sign0 Neural Network Architecture.
    Matches PyTorch checkpoint & ONNX runtime definitions across backend/app.py and export_onnx.py.
    Uses LayerNorm (batch-size invariant) for robust single-sample inference.
    """
    def __init__(self, input_dim=63, num_classes=26):
        super().__init__()
        self.in_proj = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        
        self.block1 = nn.Sequential(
            nn.Linear(256, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 256),
            nn.LayerNorm(256),
            nn.GELU()
        )
        
        self.block2 = nn.Sequential(
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 128),
            nn.LayerNorm(128),
            nn.GELU()
        )
        
        self.head = nn.Linear(128, num_classes)
        
    def forward(self, x):
        h = self.in_proj(x)
        h = h + self.block1(h) # Residual skip connection
        h2 = self.block2(h)
        return self.head(h2)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp)

train_ds = ASLLandmarkDataset(X_train, y_train, train=True)
val_ds = ASLLandmarkDataset(X_val, y_val, train=False)
test_ds = ASLLandmarkDataset(X_test, y_test, train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

model = ASLClassifierV2().to(device)
print(model)
print("Split sizes - Train:", len(train_ds), "| Val:", len(val_ds), "| Test:", len(test_ds))

## 6. Train With Early Stopping

In [ ]:
def run_epoch(loader, model, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, total_correct, total = 0.0, 0, 0

    for bx, by in loader:
        bx, by = bx.to(device), by.to(device)
        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits = model(bx)
            loss = criterion(logits, by)
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                optimizer.step()

        total_loss += loss.item() * bx.size(0)
        total_correct += (logits.argmax(1) == by).sum().item()
        total += bx.size(0)

    return total_loss / total, total_correct / total

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

best_val_acc = 0.0
bad_epochs = 0
best_path = OUTPUT_DIR / "best_asl_model.pth"
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, model, criterion, optimizer)
    scheduler.step()

    val_loss, val_acc = run_epoch(val_loader, model, criterion)
    history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc, "val_loss": val_loss, "val_acc": val_acc})

    improved = val_acc > best_val_acc
    if improved:
        best_val_acc = val_acc
        bad_epochs = 0
        torch.save(model.state_dict(), best_path)
    else:
        bad_epochs += 1

    if epoch % 5 == 0 or epoch == EPOCHS or improved:
        print(f"Epoch {epoch:03d}/{EPOCHS} | Train Acc: {train_acc*100:5.2f}% (Loss: {train_loss:.4f}) | Val Acc: {val_acc*100:5.2f}% (Loss: {val_loss:.4f}) | Best Val: {best_val_acc*100:5.2f}%")

    if bad_epochs >= PATIENCE:
        print(f"Early stopping triggered at epoch {epoch}. Best validation accuracy: {best_val_acc*100:.2f}%")
        break

print("Training finished! Best Validation Accuracy:", best_val_acc)

## 7. Final Evaluation

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()

all_targets, all_preds, all_probs = [], [], []
with torch.no_grad():
    for bx, by in test_loader:
        bx = bx.to(device)
        logits = model(bx)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_preds.append(probs.argmax(axis=1))
        all_targets.append(by.numpy())

all_probs = np.concatenate(all_probs)
all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

top1 = accuracy_score(all_targets, all_preds)
top3 = top_k_accuracy_score(all_targets, all_probs, k=3, labels=np.arange(len(LABELS)))
report = classification_report(all_targets, all_preds, target_names=LABELS, output_dict=True)

print('Top-1 accuracy:', top1)
print('Top-3 accuracy:', top3)
print(classification_report(all_targets, all_preds, target_names=LABELS))

metrics = {
    'top1_accuracy': float(top1),
    'top3_accuracy': float(top3),
    'macro_f1': float(report['macro avg']['f1-score']),
    'weighted_f1': float(report['weighted avg']['f1-score']),
    'num_train_samples': int(len(train_ds)),
    'num_val_samples': int(len(val_ds)),
    'num_test_samples': int(len(test_ds)),
    'max_images_per_class': int(MAX_IMAGES_PER_CLASS)
}

with open(OUTPUT_DIR / 'evaluation_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

cm = confusion_matrix(all_targets, all_preds, labels=np.arange(len(LABELS)))
plt.figure(figsize=(11, 9))
plt.imshow(cm, cmap='Blues')
plt.title(f'ASL Letter Confusion Matrix - Acc {top1*100:.2f}%')
plt.colorbar()
plt.xticks(np.arange(len(LABELS)), LABELS, rotation=45)
plt.yticks(np.arange(len(LABELS)), LABELS)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=160)
plt.show()

metrics

## 8. Export ONNX for FastAPI Backend

In [ ]:
onnx_path = OUTPUT_DIR / "asl_mlp.onnx"
label_map_path = OUTPUT_DIR / "label_map.json"
output_checkpoint_path = OUTPUT_DIR / "best_asl_model.pth"

model.eval().cpu()
dummy = torch.randn(1, 63, dtype=torch.float32)

try:
    torch.onnx.export(
        model,
        dummy,
        onnx_path,
        input_names=["keypoints"],
        output_names=["logits"],
        dynamic_axes={"keypoints": {0: "batch_size"}, "logits": {0: "batch_size"}},
        opset_version=14,
        do_constant_folding=True
    )
    print("ONNX export (opset 14) successful:", onnx_path)
except Exception as e:
        print(f"ONNX opset 14 export warning: {e}. Exporting with legacy opset 12...")
        torch.onnx.export(
            model,
            dummy,
            onnx_path,
            input_names=["keypoints"],
            output_names=["logits"],
            opset_version=12
        )
        print("Legacy ONNX export successful:", onnx_path)

with open(label_map_path, "w") as f:
    json.dump(LABEL2IDX, f, indent=2)

torch.save(model.state_dict(), output_checkpoint_path)

print("Saved ONNX graph:", onnx_path)
print("Saved Label Map:", label_map_path)
print("Saved Checkpoint:", output_checkpoint_path)

## 9. ONNX Sanity Check

This confirms the exported file works with `onnxruntime`, the same runtime used by your backend.

In [ ]:
session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
logits = session.run(None, {input_name: X_test[:8].astype(np.float32)})[0]
probs = np.exp(logits - logits.max(axis=1, keepdims=True))
probs = probs / probs.sum(axis=1, keepdims=True)
preds = probs.argmax(axis=1)

for i in range(len(preds)):
    print('true:', IDX2LABEL[int(y_test[i])], 'pred:', IDX2LABEL[int(preds[i])], 'conf:', float(probs[i, preds[i]]))

## 10. Download Files

Download these files and place `asl_mlp.onnx` plus `label_map.json` inside your local repo's `backend/` folder.

In [ ]:
try:
    from google.colab import files
    for path in [
        OUTPUT_DIR / "asl_mlp.onnx",
        OUTPUT_DIR / "label_map.json",
        OUTPUT_DIR / "evaluation_metrics.json",
        OUTPUT_DIR / "confusion_matrix.png",
        OUTPUT_DIR / "best_asl_model.pth"
    ]:
        if path.exists():
            print("Downloading", path.name)
            files.download(str(path))
except ImportError:
    print("Execution environment is not Google Colab. All output artifacts are saved locally in:", OUTPUT_DIR)